# PYTORCH MASTERY — NOTEBOOK 3
## Chapters 11–15: Concatenation and Stacking → Randomness and Reproducibility → Autograd → Backpropagation → Gradient Management

Continuing directly from Notebook 2. This notebook reaches PyTorch's defining capability — automatic differentiation — the mechanism that makes training a neural network possible at all.


### PYTORCH MASTERY PROGRESS
Current Chapter: 15 | Current Level: Autograd

- [x] Chapters 1–10 *(Notebooks 1–2)*
- [x] Concatenation and Stacking
- [x] Randomness and Reproducibility
- [x] Autograd
- [x] Backpropagation
- [x] Gradient Management


---
# CHAPTER 11 — CONCATENATION AND STACKING

## 1. Concept
`torch.cat()` and `torch.stack()` both combine multiple tensors into one, but they answer genuinely different questions: **concatenation joins tensors along an EXISTING dimension** (no new dimension is created — the tensors must already have compatible shapes except along the joined dimension), while **stacking creates a BRAND NEW dimension** (all input tensors must have the *identical* shape).

## 2. `cat()` vs. `stack()` — the Core Distinction, Concretely


In [ ]:
import torch

a = torch.randn(3, 4)
b = torch.randn(3, 4)

# CONCATENATION: joins along an EXISTING dimension -- no new dimension appears
cat_dim0 = torch.cat([a, b], dim=0)
print("cat along dim=0:", cat_dim0.shape, " (3+3=6 rows, still 2D)")

cat_dim1 = torch.cat([a, b], dim=1)
print("cat along dim=1:", cat_dim1.shape, " (4+4=8 columns, still 2D)")

# STACKING: creates a NEW dimension -- inputs must have IDENTICAL shape
stack_dim0 = torch.stack([a, b], dim=0)
print("\nstack along dim=0:", stack_dim0.shape, " (a NEW leading dim of size 2 appears -- now 3D!)")

stack_dim1 = torch.stack([a, b], dim=1)
print("stack along dim=1:", stack_dim1.shape, " (new dim inserted at POSITION 1 instead)")


cat along dim=0: torch.Size([6, 4])  (3+3=6 rows, still 2D)
cat along dim=1: torch.Size([3, 8])  (4+4=8 columns, still 2D)

stack along dim=0: torch.Size([2, 3, 4])  (a NEW leading dim of size 2 appears -- now 3D!)
stack along dim=1: torch.Size([3, 2, 4])  (new dim inserted at POSITION 1 instead)


### Line-by-Line Explanation
- `torch.cat([a, b], dim=0)` on two `(3, 4)` tensors produces `(6, 4)` — dimension 0 grows from 3 to `3+3=6`; dimension 1 stays `4` (it was never touched, and must already match across inputs — this is `cat`'s requirement: every dimension **except** the one being concatenated must already be identical across all inputs).
- `torch.stack([a, b], dim=0)` on two `(3, 4)` tensors produces `(2, 3, 4)` — a genuinely **new** dimension of size 2 (the number of tensors being combined) is inserted at position 0; both original tensors must have been the *exact same shape* `(3, 4)` for this to be valid at all.

## 3. The Single Most Common Real Use: Building a Batch


In [ ]:
# The canonical use case: you have a LIST of individual samples (each without a batch
# dimension) and need to combine them into ONE batched tensor for the model

individual_samples = [torch.randn(3, 32, 32) for _ in range(16)]   # 16 separate (channels, H, W) images
print("Each individual sample shape:", individual_samples[0].shape)

batch = torch.stack(individual_samples, dim=0)   # STACK, not cat -- creates the NEW batch dimension
print("Batched via stack:", batch.shape, " -- exactly (batch_size, channels, H, W)")

# Using cat() here by mistake would be WRONG -- it would try to join along an existing
# dimension, producing a nonsensical result (or an error, depending on which dim is specified)
try:
    wrong = torch.cat(individual_samples, dim=0)
    print("\ncat() 'succeeds' but gives the WRONG shape:", wrong.shape, "-- this is NOT a valid batch!")
except RuntimeError as e:
    print("cat() error:", str(e)[:150])


Each individual sample shape: torch.Size([3, 32, 32])
Batched via stack: torch.Size([16, 3, 32, 32])  -- exactly (batch_size, channels, H, W)

cat() 'succeeds' but gives the WRONG shape: torch.Size([48, 32, 32]) -- this is NOT a valid batch!


### Internal Working / Why This Distinction Matters So Much
`torch.cat([16 tensors each (3,32,32)], dim=0)` actually **succeeds** without error, but produces a single `(48, 32, 32)` tensor — it concatenated along the *channel* dimension (which happened to already exist), silently merging what should have been 16 separate images into one nonsensically-tall "image." This is a genuinely dangerous failure mode precisely *because* it doesn't crash — it silently produces a plausible-looking but completely wrong tensor, which can propagate through an entire training run before the resulting garbage performance reveals something was wrong. `torch.stack()` is the semantically correct, and typically the *safer*, choice whenever you're combining a list of same-shaped individual samples into a batch — precisely because it would raise a clear error rather than silently misinterpreting your intent if the shapes somehow mismatched.

## 4. `split()` and `chunk()` — the Inverse Operations


In [ ]:
x = torch.arange(20).reshape(10, 2)
print("x.shape:", x.shape)

# split(): specify the SIZE of each piece
split_result = torch.split(x, split_size_or_sections=3, dim=0)
print("\nsplit(size=3) produces", len(split_result), "pieces, with shapes:", [t.shape for t in split_result])
print("(10 rows / 3 per piece = 3 full pieces of 3, plus 1 remainder piece of 1)")

# chunk(): specify the NUMBER of pieces (as close to equal-sized as possible)
chunk_result = torch.chunk(x, chunks=4, dim=0)
print("\nchunk(chunks=4) produces", len(chunk_result), "pieces, with shapes:", [t.shape for t in chunk_result])


x.shape: torch.Size([10, 2])

split(size=3) produces 4 pieces, with shapes: [torch.Size([3, 2]), torch.Size([3, 2]), torch.Size([3, 2]), torch.Size([1, 2])]
(10 rows / 3 per piece = 3 full pieces of 3, plus 1 remainder piece of 1)

chunk(chunks=4) produces 4 pieces, with shapes: [torch.Size([3, 2]), torch.Size([3, 2]), torch.Size([3, 2]), torch.Size([1, 2])]


### `split` vs `chunk` — the Parameter Difference
`split(tensor, split_size, dim)` — you specify how large **each piece** should be (the last piece may be smaller if it doesn't divide evenly). `chunk(tensor, chunks, dim)` — you specify how **many pieces** total you want, and PyTorch divides as evenly as possible. Both are commonly used for splitting a batch into micro-batches (Chapter 76's gradient accumulation) or dividing a sequence into fixed-size windows.

### Common Mistakes
- Using `cat()` when you meant `stack()` (combining a list of individual, unbatched samples) — the single most damaging version of this mistake, since it can silently succeed with a wrong-but-plausible shape rather than erroring.
- Forgetting that `stack()` requires **identical** shapes across all inputs — will raise a clear error if shapes differ even slightly, unlike `cat()`'s more permissive "everything except the joined dimension must match" requirement.
- Confusing `split_size` (piece size) and `chunks` (piece count) parameters between the two functions — an easy mixup given how similar their purposes are.

### Interview Insight
**Q: You have a Python list of 32 individual image tensors, each shape `(3, 224, 224)`, and need to combine them into one batch for your model. Should you use `cat()` or `stack()`, and what happens if you use the wrong one?**
A: `stack()` — it creates a new batch dimension, producing the correct `(32, 3, 224, 224)` shape a model expects. Using `cat()` instead (e.g., with `dim=0`) would concatenate along the existing channel dimension, producing a `(96, 224, 224)` tensor that has silently merged all 32 images' channel dimensions together — this is dangerous specifically because `cat()` doesn't raise an error here; it produces a shape that's plausible enough to pass silently into the next step of a pipeline, potentially corrupting an entire training run before the resulting nonsensical performance reveals the root cause.

### Exercise
1. Create 5 tensors each of shape `(10,)` and combine them into a `(5, 10)` batch using `stack()`.
2. Given a `(100, 5)` tensor, split it into 4 roughly-equal pieces using `chunk()`, and separately into pieces of exactly 30 rows each using `split()`.

### Quiz (Chapter 11)
1. What's the fundamental difference between what `cat()` and `stack()` do to the tensor's dimensionality?
2. Why is accidentally using `cat()` instead of `stack()` a particularly dangerous mistake?
3. What's the parameter difference between `split()` and `chunk()`?


---
# CHAPTER 12 — RANDOMNESS AND REPRODUCIBILITY

## 1. Concept
Deep learning involves randomness at multiple points — weight initialization, dropout, data shuffling — and reproducing an exact result requires seeding **every** source of randomness, not just one. This directly extends the Scikit-learn course's Chapter 60 reproducibility lesson into the deep learning context, where there are meaningfully more sources of randomness to control.

## 2. `torch.manual_seed()` — the Starting Point


In [ ]:
import torch

torch.manual_seed(42)
a = torch.randn(3)
print("First run:", a)

torch.manual_seed(42)
b = torch.randn(3)
print("Second run (same seed):", b)
print("Identical:", torch.equal(a, b))


First run: tensor([0.3367, 0.1288, 0.2345])
Second run (same seed): tensor([0.3367, 0.1288, 0.2345])
Identical: True


### Every Source of Randomness You Need to Control
| Source | How to seed it |
|---|---|
| CPU tensor operations | `torch.manual_seed(seed)` |
| GPU (CUDA) tensor operations | `torch.cuda.manual_seed(seed)` (or `torch.cuda.manual_seed_all(seed)` for multi-GPU) — a **separate** random number generator from the CPU one |
| Python's own `random` module (sometimes used in data augmentation code) | `random.seed(seed)` |
| NumPy (sometimes used in `Dataset`/data pipeline code) | `np.random.seed(seed)` |
| `DataLoader` worker processes (Chapter 24) | Requires a `worker_init_fn` — each worker process has its own independent random state that the main process's seed does NOT automatically propagate to |

## 3. A Complete Reproducibility Setup


In [ ]:
import random
import numpy as np
import torch

def set_full_seed(seed: int = 42):
    """Seed EVERY source of randomness this course has identified -- the standard,
    comprehensive reproducibility setup for a real training script."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Additionally, some CUDA operations (particularly certain convolution algorithms)
    # are NON-deterministic by default even with a fixed seed, for performance reasons --
    # these two settings trade some speed for full determinism
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_full_seed(42)
print("Full reproducibility setup applied.")
print("\nNote the tradeoff: torch.backends.cudnn.benchmark=False disables cuDNN's")
print("auto-tuning (which picks the fastest convolution algorithm for your specific")
print("input shapes via brief runtime benchmarking) -- meaning training may run SLOWER")
print("but produce bit-identical results run to run, on the SAME hardware/software setup.")


Full reproducibility setup applied.

Note the tradeoff: torch.backends.cudnn.benchmark=False disables cuDNN's
auto-tuning (which picks the fastest convolution algorithm for your specific
input shapes via brief runtime benchmarking) -- meaning training may run SLOWER
but produce bit-identical results run to run, on the SAME hardware/software setup.


### Line-by-Line Explanation
- `torch.cuda.manual_seed_all(seed)` — CPU and GPU use **genuinely separate** random number generators internally; seeding one does not seed the other. `manual_seed_all` (versus `manual_seed`) additionally covers every GPU in a multi-GPU setup.
- `torch.backends.cudnn.deterministic = True` — forces cuDNN (NVIDIA's deep learning primitives library, used internally for convolutions and other operations) to use only deterministic algorithm implementations.
- `torch.backends.cudnn.benchmark = False` — disables cuDNN's autotuning, which normally benchmarks several algorithm variants at the start of training to pick the fastest one for your specific input shapes — this autotuning process itself introduces non-determinism (the "fastest" algorithm found isn't always deterministic, and the benchmarking process itself has some randomness), so disabling it is necessary for full reproducibility, at a real (sometimes meaningful) speed cost.

## 4. Why Identical Seeds STILL Don't Always Guarantee Perfect Reproducibility
Directly extending the Scikit-learn course's Chapter 60 lesson: even with every seed above set identically, bit-for-bit identical results across **different** hardware (different GPU models), different CUDA/cuDNN versions, or different PyTorch versions are **not guaranteed** — floating-point operations can accumulate in different orders depending on the specific parallel hardware/kernel implementation, producing tiny numerical differences that can compound over a long training run. "Reproducible" in practice means "reproducible within a fixed, documented hardware+software environment" — genuinely useful and worth doing, but not an absolute, universal guarantee.

### Common Mistakes
- Calling only `torch.manual_seed()` and assuming this covers GPU randomness too — it does not; CPU and GPU generators are separate.
- Forgetting `DataLoader` worker processes (Chapter 24, `num_workers > 0`) have their own independent random state, not automatically synchronized with the main process's seed — a genuinely subtle, often-missed reproducibility gap.
- Expecting bit-identical results across different GPU models or PyTorch versions even with identical seeds — a common, unrealistic expectation.
- Leaving `cudnn.benchmark = True` (the default, faster setting) while also expecting perfect run-to-run reproducibility — these two goals are directly in tension.

### Interview Insight
**Q: You set `torch.manual_seed(42)` at the start of your training script, but two separate runs on the same GPU still produce slightly different final model weights. What are you likely missing?**
A: `torch.manual_seed()` only seeds the CPU random number generator — GPU (CUDA) operations use a separate generator requiring `torch.cuda.manual_seed_all()`. Additionally, if using a `DataLoader` with `num_workers > 0`, each worker process has independent randomness not covered by the main process's seed unless explicitly handled via `worker_init_fn`. And even with every generator seeded, `torch.backends.cudnn.benchmark=True` (PyTorch's default) allows cuDNN's autotuning process to introduce non-determinism in convolution algorithm selection — setting `cudnn.deterministic=True` and `cudnn.benchmark=False` is required for full determinism, at some cost to training speed.

### Exercise
1. Write your own `set_full_seed()` function from memory, including every source of randomness this chapter identified.
2. Explain, in your own words, the specific tradeoff `cudnn.benchmark=False` makes.

### Quiz (Chapter 12)
1. Name all 5 distinct sources of randomness a comprehensive reproducibility setup needs to control.
2. Why doesn't `torch.manual_seed()` alone cover GPU randomness?
3. What does `torch.backends.cudnn.benchmark=True` (the default) trade off against reproducibility?
4. Why can identical seeds still fail to produce bit-identical results across different hardware?


---
# CHAPTER 13 — AUTOGRAD

## 1. Concept
Autograd is PyTorch's **automatic differentiation engine** — the single capability that makes training a neural network practical without manually deriving and coding every gradient formula by hand. As you perform operations on tensors with `requires_grad=True`, PyTorch silently builds a **computational graph** recording exactly what operations happened and in what order; calling `.backward()` on a final scalar output then walks this graph backward, applying the chain rule automatically at every step, to compute the gradient of that output with respect to every tensor that contributed to it.

## 2. Why It Exists — The Problem It Solves
Before automatic differentiation frameworks, implementing a new neural network architecture meant manually deriving the gradient of the loss with respect to every single parameter — a genuinely error-prone, time-consuming process that scaled terribly as architectures grew more complex (imagine manually deriving gradients for a 100-layer network, or for a novel attention mechanism). Autograd removes this entirely: you write the **forward** computation naturally, in plain Python/PyTorch, and the **backward** (gradient) computation is derived automatically, correctly, every time — even for architectures involving loops, conditionals, and arbitrary custom operations.

## 3. The Simplest Possible Example


In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)   # a LEAF tensor -- something we want a gradient WITH RESPECT TO
y = x ** 2                                   # y = x^2

y.backward()                                 # compute dy/dx

print("x:", x.item())
print("y:", y.item())
print("x.grad (dy/dx):", x.grad.item())
print("\nMathematically: y = x^2, so dy/dx = 2x. At x=2, dy/dx = 2*2 = 4. Matches!")


x: 2.0
y: 4.0
x.grad (dy/dx): 4.0

Mathematically: y = x^2, so dy/dx = 2x. At x=2, dy/dx = 2*2 = 4. Matches!


### Line-by-Line Explanation
- `torch.tensor(2.0, requires_grad=True)` — creates a **leaf tensor**: a tensor that is a genuine starting point of the computational graph (not itself derived from other `requires_grad=True` tensors) and for which PyTorch will accumulate a `.grad` attribute when `.backward()` is called. Model parameters (weights, biases) are the canonical example of leaf tensors in real training code.
- `y = x ** 2` — this operation is recorded onto a computational graph **automatically**, purely because `x.requires_grad=True`; `y` itself now also has `requires_grad=True` (inherited) and additionally a `.grad_fn` attribute recording *how* it was computed (here, a power operation), which is what `.backward()` uses to know how to differentiate it.
- `y.backward()` — triggers the backward pass: PyTorch walks the recorded graph from `y` back to every leaf tensor that contributed to it, applying the chain rule at each recorded operation, and **accumulates** the resulting gradient into each leaf's `.grad` attribute (`x.grad`, here).
- `x.grad` starts as `None` until the first `.backward()` call populates it — a common source of `AttributeError: 'NoneType' object has no attribute ...` for a tensor that either never had `requires_grad=True` set, or was never actually part of the graph that `.backward()` was called on.

## 4. A Slightly Richer Example — Multiple Variables and the Chain Rule


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b        # y = w*x + b -- exactly a single-feature linear layer's forward computation
loss = (y - 10) ** 2  # a made-up "loss": squared error against a target of 10

loss.backward()

print(f"x={x.item()}, w={w.item()}, b={b.item()}")
print(f"y = w*x + b = {y.item()}")
print(f"loss = (y - 10)^2 = {loss.item()}")
print()
print("Gradients computed automatically via the chain rule:")
print("dloss/dw:", w.grad.item())
print("dloss/db:", b.grad.item())
print("dloss/dx:", x.grad.item())

print("""
Verifying dloss/dw by hand:
  loss = (w*x + b - 10)^2
  Let u = w*x + b - 10 = (2*3 + 1 - 10) = -3
  dloss/du = 2u = -6
  du/dw = x = 3
  dloss/dw = dloss/du * du/dw = -6 * 3 = -18   <- matches w.grad exactly, via the chain rule
""")


x=3.0, w=2.0, b=1.0
y = w*x + b = 7.0
loss = (y - 10)^2 = 9.0

Gradients computed automatically via the chain rule:
dloss/dw: -18.0
dloss/db: -6.0
dloss/dx: -12.0

Verifying dloss/dw by hand:
  loss = (w*x + b - 10)^2
  Let u = w*x + b - 10 = (2*3 + 1 - 10) = -3
  dloss/du = 2u = -6
  du/dw = x = 3
  dloss/dw = dloss/du * du/dw = -6 * 3 = -18   <- matches w.grad exactly, via the chain rule



### Internal Working — What the Computational Graph Actually Looks Like
```
x (leaf) ──┐
           ├──> [mul] ──> w*x ──┐
w (leaf) ──┘                    ├──> [add] ──> y ──> [sub 10] ──> [pow 2] ──> loss
b (leaf) ────────────────────────┘
```
Every intermediate node stores a reference to the operation that produced it (its `.grad_fn`) and to its inputs. `.backward()` performs a **reverse-mode** traversal: starting from `loss`, it computes `d(loss)/d(loss) = 1`, then propagates backward through each operation using that operation's known local derivative (e.g., `pow(2)`'s local derivative is `2*input`), multiplying along the way (the chain rule) until it reaches every leaf tensor, depositing the final accumulated gradient into each leaf's `.grad`.

### `retain_graph` and Calling `.backward()` Twice


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

y.backward()
print("First backward() -- x.grad:", x.grad.item())

try:
    y.backward()   # calling backward() a SECOND time on the SAME graph
except RuntimeError as e:
    print("\nSecond backward() FAILS:", str(e)[:150])

print("""
By default, PyTorch FREES the computational graph's intermediate buffers immediately
after backward() runs, to save memory -- this is why a second call fails. If you
genuinely need to call backward() multiple times on the same graph (a rare, advanced
need), pass retain_graph=True to the FIRST call.
""")


First backward() -- x.grad: 4.0

Second backward() FAILS: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of

By default, PyTorch FREES the computational graph's intermediate buffers immediately
after backward() runs, to save memory -- this is why a second call fails. If you
genuinely need to call backward() multiple times on the same graph (a rare, advanced
need), pass retain_graph=True to the FIRST call.



### Common Mistakes
- Forgetting `requires_grad=True` on a tensor and then being confused why `.grad` is `None` after `.backward()` — only leaf tensors with `requires_grad=True` (and tensors derived from them) accumulate gradients at all.
- Calling `.backward()` on a **non-scalar** tensor without providing a `gradient` argument — `.backward()`'s default behavior computes the gradient of a scalar output; for a non-scalar `y`, you must either first reduce it to a scalar (e.g., `.sum()`) or explicitly pass a `gradient=` argument specifying the "upstream" gradient to backpropagate from — a genuinely common early error (`RuntimeError: grad can be implicitly created only for scalar outputs`).
- Not realizing gradients **accumulate** by default across multiple `.backward()` calls (Chapter 15's `zero_grad()` exists specifically to address this) — calling `.backward()` twice without zeroing gradients in between adds the two gradients together, not replacing the first with the second.

### Interview Insight
**Q: Explain, at a mechanical level, what happens when you call `.backward()` on a scalar tensor that resulted from several chained operations on `requires_grad=True` inputs.**
A: As the forward computation executes, PyTorch builds a computational graph on the fly — every operation on a `requires_grad=True` tensor (or a tensor derived from one) creates a new graph node recording that operation and references to its inputs. Calling `.backward()` on the final scalar output triggers a reverse-mode traversal of this graph: PyTorch starts with a gradient of `1` at the output, then walks backward through each recorded operation, applying that operation's known local derivative formula and the chain rule to propagate the gradient backward, accumulating the final result into the `.grad` attribute of every leaf tensor (tensors with `requires_grad=True` that weren't themselves derived from other tracked tensors) that contributed to the output.

### Exercise
1. Create `x = torch.tensor(5.0, requires_grad=True)`, compute `y = 3*x**3 + 2*x`, call `.backward()`, and verify `x.grad` matches the hand-computed derivative `9*x**2 + 2` at `x=5`.
2. Deliberately try to call `.backward()` twice on the same graph without `retain_graph=True` and observe the error.

### Quiz (Chapter 13)
1. What is a "leaf tensor," and why does it matter for where gradients get stored?
2. What does `.grad_fn` on a non-leaf tensor represent?
3. Why does calling `.backward()` a second time on the same graph fail by default, and what parameter fixes this?
4. What must you do before calling `.backward()` on a non-scalar tensor?


---
# CHAPTER 14 — BACKPROPAGATION

## 1. Concept
Backpropagation is the specific algorithm autograd (Chapter 13) implements — the general recipe for training any neural network:

```
Forward Pass         -- compute predictions from inputs, layer by layer
     ↓
Loss                 -- compare predictions to true targets with a loss function
     ↓
Computational Graph   -- PyTorch has been recording every operation along the way
     ↓
Backward Pass         -- loss.backward() walks the graph in reverse, via the chain rule
     ↓
Gradients             -- every parameter now has a .grad telling us "which direction increases the loss"
     ↓
Parameter Update      -- the optimizer (Chapter 20) nudges every parameter OPPOSITE its gradient
```

## 2. The Chain Rule, Explained From First Principles (No Calculus Assumed)
The chain rule answers: "if I know how a small change in `A` affects `B`, and how a small change in `B` affects `C`, how does a small change in `A` affect `C`?" Answer: **multiply the two individual effects together**. Formally, for `C = f(B)` and `B = g(A)`:
```
dC/dA = dC/dB * dB/dA
```
A neural network's forward pass is a long chain of exactly this kind of composition — raw input → layer 1's output → layer 2's output → ... → final loss. Backpropagation is nothing more than applying this multiplication rule repeatedly, once per layer, walking backward from the loss to every single parameter in the network — which is *exactly* what autograd's computational-graph traversal (Chapter 13) automates.

## 3. Manually Implementing Backprop for a Tiny Network (to Build True Intuition)


In [ ]:
import torch

# A TWO-layer network, by hand, to see EVERY step explicitly before trusting autograd to do it

# Manual forward pass
x = torch.tensor([1.0, 2.0])                          # a single input sample, 2 features
w1 = torch.tensor([[0.5, -0.3], [0.2, 0.8]], requires_grad=True)   # layer 1 weights, 2x2
b1 = torch.tensor([0.1, -0.1], requires_grad=True)
w2 = torch.tensor([0.4, -0.6], requires_grad=True)      # layer 2 weights, maps 2 -> 1
b2 = torch.tensor(0.05, requires_grad=True)

h = torch.relu(x @ w1 + b1)    # layer 1: linear + ReLU activation
y_pred = h @ w2 + b2           # layer 2: linear (no activation -- a regression output)

target = torch.tensor(1.5)
loss = (y_pred - target) ** 2   # mean squared error, for a single sample

print("h (hidden layer output):", h)
print("y_pred:", y_pred.item())
print("loss:", loss.item())

# NOW let autograd do the backward pass for us
loss.backward()

print("\nGradients computed automatically:")
print("w1.grad:\n", w1.grad)
print("b1.grad:", b1.grad)
print("w2.grad:", w2.grad)
print("b2.grad:", b2.grad.item())


h (hidden layer output): tensor([1.0000, 1.2000], grad_fn=<ReluBackward0>)
y_pred: -0.2699999511241913
loss: 3.1328999996185303

Gradients computed automatically:
w1.grad:
 tensor([[-1.4160,  2.1240],
        [-2.8320,  4.2480]])
b1.grad: tensor([-1.4160,  2.1240])
w2.grad: tensor([-3.5400, -4.2480])
b2.grad: -3.5399999618530273


### Line-by-Line Explanation — Connecting This to Real Model Training
- This IS, structurally, exactly what happens inside every `nn.Module`-based network's training step (Chapter 16 onward) — `nn.Linear` layers replace the manual `@` and `+` operations, and `nn.ReLU()` replaces the manual `torch.relu()` call, but the underlying forward-pass-then-`.backward()` mechanism is identical.
- Every gradient (`w1.grad`, `b1.grad`, `w2.grad`, `b2.grad`) was computed by a **single** `loss.backward()` call, walking the graph all the way from the scalar `loss` back through `y_pred`, `h`, to every leaf parameter — you never had to manually derive or code a single derivative formula, no matter how many layers were involved.
- Notice `w2.grad` is affected by `h` (through the chain rule: `d(loss)/d(w2)` depends on `h`, since `y_pred = h @ w2 + b2`), and `w1.grad` is affected by `w2` too (through `h`'s influence on `y_pred`, and `y_pred`'s influence on `loss`) — this cascading, multiplicative dependency across every layer is precisely what "backpropagation" refers to: gradients flow **backward** through the network, layer by layer, each layer's gradient depending on the gradient of the layer *after* it.

## 4. What "Vanishing" and "Exploding" Gradients Mean, Concretely


In [ ]:
# Simulating a DEEP chain of small multiplications -- illustrating vanishing gradients conceptually
small_factor = torch.tensor(0.5)
product = torch.tensor(1.0)
for layer in range(20):
    product = product * small_factor
print(f"After 20 layers, each multiplying by 0.5: {product.item():.10f}")
print("This is EXACTLY what happens to a gradient flowing backward through many layers")
print("if each layer's LOCAL derivative is consistently < 1 -- the gradient shrinks")
print("MULTIPLICATIVELY, layer by layer, and can become numerically indistinguishable from")
print("zero by the time it reaches early layers -- those early layers then barely learn at all.")

large_factor = torch.tensor(1.5)
product2 = torch.tensor(1.0)
for layer in range(20):
    product2 = product2 * large_factor
print(f"\nAfter 20 layers, each multiplying by 1.5: {product2.item():.2f}")
print("The OPPOSITE failure -- gradients EXPLODE multiplicatively, potentially overflowing")
print("to inf/NaN (Chapter 33) and destroying training entirely.")


After 20 layers, each multiplying by 0.5: 0.0000009537
This is EXACTLY what happens to a gradient flowing backward through many layers
if each layer's LOCAL derivative is consistently < 1 -- the gradient shrinks
MULTIPLICATIVELY, layer by layer, and can become numerically indistinguishable from
zero by the time it reaches early layers -- those early layers then barely learn at all.

After 20 layers, each multiplying by 1.5: 3325.26
The OPPOSITE failure -- gradients EXPLODE multiplicatively, potentially overflowing
to inf/NaN (Chapter 33) and destroying training entirely.


### Internal Working / Why This Motivates Real Architectural Choices
Vanishing/exploding gradients are a direct, mechanical consequence of the chain rule's *multiplicative* nature applied across many layers — this single insight directly motivates several major deep learning innovations covered later in this course: ReLU activations (Chapter 18) specifically avoid the "saturating" gradient regions that sigmoid/tanh suffer from; residual/skip connections (Chapter 39's ResNet) give gradients a direct additive path around multiplicative chains; LSTM/GRU gating mechanisms (Chapters 45–46) were specifically designed to combat vanishing gradients in recurrent networks; and gradient clipping (Chapter 77) directly caps the exploding-gradient failure mode.

### Common Mistakes
- Building extremely deep networks with saturating activations (sigmoid/tanh) throughout and being surprised early layers don't learn — a direct, textbook vanishing-gradient symptom.
- Not recognizing a `NaN` loss partway through training as a likely gradient-explosion symptom — Chapter 33's debugging chapter covers this pattern explicitly.
- Assuming backpropagation is a separate algorithm you need to implement — in PyTorch, `.backward()` *is* backpropagation, automated; you should almost never need to manually derive gradients for a standard architecture.

### Interview Insight
**Q: Explain, mechanically, why very deep networks historically suffered from vanishing gradients, and name two modern techniques that address this.**
A: Backpropagation computes each parameter's gradient via the chain rule — a product of each layer's local derivative, multiplied together across every layer between that parameter and the loss. If each layer's local derivative is consistently less than 1 in magnitude (a common situation with saturating activations like sigmoid, whose derivative maxes out at 0.25), this product shrinks multiplicatively with network depth, becoming numerically negligible by the time it reaches early layers — those layers then receive almost no learning signal. Residual/skip connections (ResNet, Chapter 39) address this by providing an additive path for gradients to flow directly backward, bypassing the multiplicative chain; ReLU activations (Chapter 18) avoid sigmoid/tanh's severe gradient saturation in their active region; and LSTM/GRU gating (Chapters 45–46) was specifically designed to let gradients flow through recurrent time steps without the same multiplicative decay a plain RNN suffers from.

### Exercise
1. Manually implement a 3-layer network's forward pass (linear → ReLU → linear → ReLU → linear) using raw tensor operations and `requires_grad=True`, compute an MSE loss, and call `.backward()`.
2. Print every layer's gradient and verify none are exactly zero or absurdly large (a quick, informal vanishing/exploding gradient sanity check).

### Quiz (Chapter 14)
1. State the chain rule in your own words, without calculus jargon.
2. Why does backpropagation compute gradients in reverse order (loss → early layers), rather than forward?
3. Mechanically, why does a "vanishing gradient" happen in a deep network with saturating activations?
4. Name three architectural techniques this course will cover that directly address vanishing/exploding gradients.


---
# CHAPTER 15 — GRADIENT MANAGEMENT

## 1. Concept
Beyond the core `.backward()` mechanism, professional PyTorch code requires precise control over **when** gradients are tracked, **when** they're cleared, and **when** they're deliberately detached from the graph — getting any of these wrong is a leading cause of training bugs (Chapter 33).

## 2. `zero_grad()` — Why Gradients Must Be Explicitly Cleared Every Step


In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("After first backward():", x.grad.item())   # 2*2 = 4

y2 = x ** 2
y2.backward()
print("After SECOND backward() -- WITHOUT zeroing first:", x.grad.item())
print("(4 + 4 = 8 -- gradients ACCUMULATED, they did NOT reset!)")

# THE FIX: explicitly zero before each new backward pass
x.grad = None   # or x.grad.zero_()
y3 = x ** 2
y3.backward()
print("\nAfter zeroing and a fresh backward():", x.grad.item(), " (correctly back to 4)")


After first backward(): 4.0
After SECOND backward() -- WITHOUT zeroing first: 8.0
(4 + 4 = 8 -- gradients ACCUMULATED, they did NOT reset!)

After zeroing and a fresh backward(): 4.0  (correctly back to 4)


### Why PyTorch Accumulates Gradients by Default (Rather Than Resetting Automatically)
This is a deliberate design choice, not an oversight — gradient **accumulation** is itself a genuinely useful technique (Chapter 76: simulating a larger batch size by summing gradients across several smaller "micro-batches" before taking one optimizer step). Making accumulation the default behavior, with explicit zeroing required, gives you this capability "for free" — the cost is that **every standard training loop must remember to call `zero_grad()`** at the start of each iteration, or gradients silently accumulate across steps in a way that's almost never what you actually want for standard training.

In real training code, this is virtually always: `optimizer.zero_grad()` (Chapter 20) — zeroing every parameter the optimizer is tracking in one call, rather than doing it tensor-by-tensor as shown above for illustration.

## 3. `torch.no_grad()` vs. `torch.inference_mode()` — the Precise Difference


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# WITHOUT any context manager -- gradient tracking happens (and costs memory/compute) even
# if you never intend to call backward()
y_tracked = x ** 2
print("Without no_grad(): y.requires_grad =", y_tracked.requires_grad)

# torch.no_grad() -- the classic way to disable gradient tracking for a block of code
with torch.no_grad():
    y_no_grad = x ** 2
    print("Inside no_grad(): y.requires_grad =", y_no_grad.requires_grad)

# torch.inference_mode() -- PyTorch's NEWER, STRICTER, and generally FASTER alternative
with torch.inference_mode():
    y_inference = x ** 2
    print("Inside inference_mode(): y.requires_grad =", y_inference.requires_grad)

# The KEY difference: a tensor created inside inference_mode() CANNOT be used later
# in a computation that requires grad tracking, even OUTSIDE the context manager --
# no_grad()'s outputs CAN still (in limited ways) re-enter a tracked graph later


Without no_grad(): y.requires_grad = True
Inside no_grad(): y.requires_grad = False
Inside inference_mode(): y.requires_grad = False


### The Precise Distinction, and Why It Matters
Both context managers disable gradient tracking for the code inside them, producing tensors with `requires_grad=False`, and both save memory/computation by skipping the construction of a computational graph. `torch.inference_mode()` (introduced in PyTorch 1.9+) goes further: it makes additional internal optimizations specifically safe because it guarantees tensors created inside it will **never** need gradient tracking, even later — this extra guarantee lets PyTorch skip additional bookkeeping `no_grad()` still performs "just in case," making `inference_mode()` measurably faster for pure inference. The practical rule: **use `torch.inference_mode()` for genuine inference/evaluation code where you're certain no gradient will ever be needed on the results; use `torch.no_grad()` in more mixed contexts** (some older codebases, or contexts requiring inference_mode's stricter guarantee to be relaxed).

## 4. `.detach()` — Removing a Tensor From the Graph, Selectively


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = y.detach()   # z shares y's DATA but is disconnected from the computational graph

print("y.requires_grad:", y.requires_grad)
print("z.requires_grad:", z.requires_grad, " (detached -- no longer tracked)")
print("z shares y's underlying data:", z.data_ptr() == y.data_ptr())

# A genuinely common real use: logging/plotting a loss value during training WITHOUT
# keeping its entire computational graph alive in memory unnecessarily
loss_value_for_logging = y.detach().item()   # .item() ALSO implicitly detaches + converts to a Python number
print("\nSafely extracted for logging:", loss_value_for_logging)


y.requires_grad: True
z.requires_grad: False  (detached -- no longer tracked)
z shares y's underlying data: True

Safely extracted for logging: 4.0


### `no_grad()` vs. `detach()` — When to Use Each
- **`torch.no_grad()`/`torch.inference_mode()`** — a context manager wrapping a **block of code**; use during evaluation/inference, where you know an entire section of computation never needs gradients.
- **`.detach()`** — operates on a **single tensor**; use when you need to "break off" one specific tensor from an ongoing gradient-tracked computation (e.g., using a model's output as a fixed target for another loss, without backpropagating through the model that produced it — a genuinely common pattern in techniques like GANs, Chapter 63).

### Common Mistakes
- Forgetting `optimizer.zero_grad()` at the start of each training iteration — gradients silently accumulate across steps, corrupting training in a way that often doesn't crash, just produces subtly (or badly) wrong results.
- Running inference/evaluation code **without** `torch.no_grad()`/`inference_mode()` — wastes substantial memory (an entire unnecessary computational graph is built and retained) and compute, a genuine, measurable performance mistake in production inference code.
- Confusing `.detach()` (operates on one tensor, doesn't prevent future operations elsewhere from tracking gradients) with `torch.no_grad()` (a block-wide context that prevents gradient tracking for everything inside it).
- Calling `.item()` (which implicitly detaches) inside a loop expecting to still backpropagate through that value later — `.item()` returns a plain Python number, permanently disconnected from any graph.

### Interview Insight
**Q: What's the practical difference between `torch.no_grad()` and `torch.inference_mode()`, and when would you choose one over the other?**
A: Both disable gradient tracking for the code within them, producing `requires_grad=False` tensors and saving the memory/compute of building a computational graph. `torch.inference_mode()` additionally guarantees that tensors created inside it can never later be used in a gradient-tracked computation, even after exiting the context — this stronger guarantee lets PyTorch skip extra internal bookkeeping that `no_grad()` still performs, making `inference_mode()` measurably faster. The practical guidance: use `torch.inference_mode()` for genuine model evaluation/inference code (the common, default choice in modern PyTorch), and reserve `torch.no_grad()` for contexts needing its slightly more permissive behavior.

### Exercise
1. Write a mini training loop (5 iterations) that deliberately forgets `zero_grad()`, and observe the gradient values growing across iterations. Then fix it.
2. Write a small inference function using `torch.inference_mode()` correctly, extracting a final prediction as a plain Python value via `.item()`.

### Quiz (Chapter 15)
1. Why does PyTorch accumulate gradients by default instead of resetting them automatically after every `.backward()` call?
2. What's the precise, stronger guarantee `torch.inference_mode()` provides over `torch.no_grad()`?
3. What's the difference in scope between `.detach()` and `torch.no_grad()`?
4. What does `.item()` do to a tensor's connection to the computational graph?

### Chapter 15 Summary
- `zero_grad()` (almost always called via the optimizer, Chapter 20) must run every training iteration, or gradients silently accumulate incorrectly.
- `torch.inference_mode()` is the modern, preferred default for evaluation/inference code; `torch.no_grad()` remains valid for more mixed contexts.
- `.detach()` selectively disconnects one tensor from the graph; `.item()` implicitly detaches and converts to a plain Python number.

---

## END OF NOTEBOOK 3 — What You Should Now Be Able To Do
- Correctly distinguish `cat()` (existing dimension) from `stack()` (new dimension) and know why confusing them is dangerous.
- Set up complete, multi-source reproducibility (CPU, GPU, Python, NumPy, cuDNN determinism) and explain its real limits.
- Explain autograd's computational graph mechanism precisely: leaf tensors, `.grad_fn`, and what `.backward()` actually does.
- Explain backpropagation as the chain rule applied layer by layer, and explain vanishing/exploding gradients mechanically.
- Manage gradients correctly in real training/inference code: `zero_grad()` every iteration, `inference_mode()` for evaluation, `.detach()` for selectively breaking the graph.

## Before Moving to Notebook 4 (Chapters 16–20: nn.Module, Layers, Activation Functions, Loss Functions, Optimizers)
 